# 52. 蜂群图（swarmplot）

<!-- module-learning-arc:start -->
> **Seaborn 模块主线｜第 9 / 20 步：比较类别频数、水平与组内分布**
>
> **持续应用背景：** 开展客群消费行为差异研究：先固定样本和统计语义，再比较分布、关系和分面结果，判断差异是否稳定。
>
> **承接上一阶段：** 抖动散点图（stripplot）  →  **本章任务：** 蜂群图（swarmplot）  →  **下一步：** 直方图（histplot）
>
> **大作业连接：** 本章练习将成为《客群消费行为差异研究》的一部分，最终需要从样本口径和分布比较走到关系验证、分面研究与因果边界说明。
<!-- module-learning-arc:end -->


## 本章场景

当你面对“类别 + 数值”的数据，比如各品类的订单金额，最关心的往往是每个品类内部数值摊得有多开、是集中还是分散。



## 本章目标

学完本章，你将能够：

- **理解**：理解「蜂群图（swarmplot）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「蜂群图（swarmplot）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「蜂群图（swarmplot）」并读出其中的结论。


## 52.1 适用场景

**背景引入**：当你面对“类别 + 数值”的数据，比如各品类的订单金额，最关心的往往是每个品类内部数值摊得有多开、是集中还是分散。普通散点图数据一多就挤成一团，看不出分布；蜂群图会按数值高低把每个点轻轻“推开”，既保留每一个原始点，又能看清整条分布轮廓，非常适合样本量不大、想同时观察分布和离群情况的场景。

样本量不大，需要避免点重叠并观察实际分布。


## 52.2 数据结构

分类变量与数值变量；通常每组不超过数百点。


## 52.3 本章练习任务

运行基础图表后，完成以下任务：

1. 将 size 参数从 4 改为 6 或 2，观察点大小对蜂群布局密度的影响
2. 修改 dodge=True 为 dodge=False，对比分组错位与叠加的横向展开效果
3. 移除 violinplot 背景层，只保留 swarmplot，说明蜂群图单独使用时的信息完整性


## 52.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `orders.sample()`、`plt.subplots()`、`sns.swarmplot()`、`ax.set()` | 样本量不大，需要避免点重叠并观察实际分布。 | 大样本渲染慢 |
| 进阶变体 | `orders.sample()`、`plt.subplots()`、`sns.violinplot()`、`sns.swarmplot()` | 在基础图表上增加分组、注释、布局或交互 | 点过大无法完成布局 |
| 关键参数 | `size` | 点大小 | 大样本渲染慢 |
| 关键参数 | `hue` | 分组 | 点过大无法完成布局 |
| 关键参数 | `dodge` | 错位 | 把横向位置当成数据值 |
| 关键参数 | `warn_thresh` | 拥挤警告 | 大样本渲染慢 |


## 52.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，即使 seaborn 的 sns.set_theme
#      会重置字体，运行时也会在 set_theme 之后自动恢复。因此这里无需手动
#      import 或 addfont，直接使用即可。

# 1️⃣ 主题与数据导入：统一画风，读取三个公开数据集
sns.set_theme(style="whitegrid", context="notebook")

diamonds = pd.read_csv("/datasets/diamonds.csv")
taxis = pd.read_csv("/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
flights = pd.read_csv("/datasets/flights.csv")
print(f"Diamonds {len(diamonds):,} | Taxis {len(taxis):,} | Flights {len(flights):,} 行")


In [ ]:
# 2️⃣ 特征工程：把原始字段映射成图表统一使用的列名与派生指标
orders_full = diamonds.assign(
    category=diamonds["cut"],
    channel=diamonds["color"],
    region=diamonds["clarity"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    satisfied=np.where(
        diamonds["price"] >= diamonds["price"].median(), "高于中位价", "不高于中位价"
    ),
)
orders = orders_full.sample(2_000, random_state=36)

marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"),
    visits=taxis["distance"],
    ad_spend=taxis["tip"],
    sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(
    min(2_000, len(marketing_full)), random_state=36
).copy()

daily = flights.assign(
    date=pd.to_datetime(
        flights["year"].astype(str) + "-" + flights["month"] + "-01"
    ),
    region="AirPassengers",
    sales=flights["passengers"],
)
print(f"样本：orders {len(orders):,} | marketing {len(marketing):,} | daily {len(daily):,} 行")


## 52.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

sample = orders.sample(90, random_state=43)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.swarmplot(
    data=sample,
    x="category",
    y="order_value",
    hue="category",
    palette="Set2",
    legend=False,
    size=4,
    ax=ax,
)
ax.set(title="品类客单价蜂群图", xlabel="品类", ylabel="客单价（元）")
fig.tight_layout()
plt.show()


**练一练**：还记得 46.4 基础图表里的 `size=4` 吗？现在请你把点大小参数改成 6（也可以试试 2），重新绘制“品类客单价蜂群图”，然后对比：点变大后，同一品类内部点是更难还是更容易看清？最后统计每个品类各有多少个样本。


In [ ]:
# 请在下方填写代码：把 46.4 基础图表的点大小参数 size 改为 6，再统计每个品类的样本量。
# 提示：swarmplot 的点大小由 size=... 控制；样本量用 df.groupby("列名").size()。


In [ ]:
# ===== Python Data Studio: CJK font support =====
# 解题：把点大小参数 size 由 4 改为 6，点的视觉权重变大、重叠区更明显；
# 再用数量统计确认每个品类都保留了足够样本，便于横向比较分布。

sample = orders.sample(90, random_state=43)
size_value = 6  # 1) 修改点大小参数
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.swarmplot(
    data=sample,
    x="category",
    y="order_value",
    hue="category",
    palette="Set2",
    legend=False,
    size=size_value,
    ax=ax,
)
ax.set(title="品类客单价蜂群图（size=6）", xlabel="品类", ylabel="客单价（元）")
fig.tight_layout()
plt.show()


## 52.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

sample = orders.sample(120, random_state=430)
fig, ax = plt.subplots(figsize=(9, 4.8))
sns.violinplot(
    data=sample,
    x="category",
    y="order_value",
    color="#e8eaed",
    inner=None,
    cut=0,
    ax=ax,
)
sns.swarmplot(
    data=sample,
    x="category",
    y="order_value",
    hue="satisfied",
    palette=["#188038", "#f9ab00"],
    size=3.5,
    ax=ax,
)
ax.set(title="密度轮廓与真实订单", xlabel="品类", ylabel="客单价（元）")
ax.legend(title="评价", frameon=False)
fig.tight_layout()
plt.show()


## 52.8 参数说明

- size：点大小
- hue：分组
- dodge：错位
- warn_thresh：拥挤警告


## 52.9 结果解读

横向展开宽度反映局部点密度，每个点仍是一条真实观察。


## 52.10 本章实训：分组比较与不确定性

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="region", y="sales", ci=None, ax=ax, color="#0F766E"
)
ax.set_title("地区销售额比较")
ax.set_ylabel("销售额")
plt.show()


### 52.10.1 第一个结果怎么读

Seaborn 负责把 DataFrame 的字段映射为图形编码；先明确横轴、纵轴和每行数据的粒度，再选择图表。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
report = report.sort_values("sales", ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="sales", y="region", ci=None, ax=ax, color="#F59E0B"
)
ax.set_title("按销售额排序的地区比较")
ax.set_xlabel("销售额")
ax.set_ylabel("地区")
plt.show()


### 52.10.2 第二个结果怎么读

第二个实验只改变排序和坐标方向，让读者更容易找到最大值。图表调整必须服务于阅读任务。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 52.11 错误恢复：分组字段缺失怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
required = {"region", "sales"}
missing = required - set(report.columns)
if missing:
    print("缺少字段：", sorted(missing))
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.barplot(data=report, x="region", y="sales", ci=None, ax=ax)
    ax.set_title("地区销售额")
    plt.show()


### 52.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

绘图前先检查字段是否存在。把字段检查放在画图之前，错误会更接近真正原因，也更容易恢复。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 52.12 易错点提醒

- 大样本渲染慢
- 点过大无法完成布局
- 把横向位置当成数据值


## 52.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 52.14 独立迁移练习

修改一个分组、排序或统计设置，并比较修改前后的结论。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：把蜂群图与箱线图叠加，结合看分布与离群点
# 【目标】蜂群看原始点、箱线看摘要，二者叠加信息互补。
import matplotlib.pyplot as plt
import seaborn as sns

# 起点示例(已可运行)：先画蜂群(灰点)，再叠一层箱线(白箱)观察两者关系。
sample = orders.sample(90, random_state=43)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.swarmplot(
    data=sample, x="category", y="order_value", color="#9aa0a6", size=4, ax=ax
)
sns.boxplot(
    data=sample, x="category", y="order_value", color="#ffffff", width=0.5, ax=ax
)
ax.set(title="蜂群图叠加箱线", xlabel="品类", ylabel="客单价（元）")
fig.tight_layout()
plt.show()

# ---- 反思记录：叠加之后，箱线摘要与原始点如何互相印证 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
import matplotlib.pyplot as plt

sample = orders.sample(75, random_state=431)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.swarmplot(
    data=sample,
    x="region",
    y="items",
    hue="channel",
    dodge=True,
    size=4,
    palette="colorblind",
    ax=ax,
)
ax.set(title="区域购买件数蜂群图", xlabel="区域", ylabel="件数")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()


## 52.15 小结

用无重叠蜂群布局展示分类组内的原始点和局部密度。


### 52.15.1 你已经掌握

- 判断蜂群图（swarmplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 52.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `size` | 点大小 |
| `hue` | 分组 |
| `dodge` | 错位 |
| `warn_thresh` | 拥挤警告 |


### 52.15.3 需要注意

- 大样本渲染慢
- 点过大无法完成布局
- 把横向位置当成数据值


### 52.15.4 完成检查

- [ ] 能判断什么问题适合使用蜂群图（swarmplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 52.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
